In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from skfuzzy import control as ctrl

import random
import copy


In [8]:
file_path = r"..\data\processed\healthcare_processed.csv"
fuzzy_data = pd.read_csv(file_path)
# fuzzy_data.head()

# Correct feature set for fuzzy-GA system
X_fuzzy = fuzzy_data[['age', 'bmi','gender','ever_married',
                      'work_type','smoking_status','avg_glucose_level',
                      'hypertension', 'heart_disease', 'Residence_type']]

y_fuzzy = fuzzy_data['stroke_risk']  # target classes: 0=Low, 1=Medium, 2=High

print("X shape:", X_fuzzy.shape)
print("y shape:", y_fuzzy.shape)


X shape: (5110, 10)
y shape: (5110,)


In [9]:
# Notebook 4 — load spec and rebuild fuzzy system
import pickle
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

with open(r"..\\data\\processed\\fuzzy_system_spec.pkl", "rb") as f:
    spec = pickle.load(f)

# Helper to create antecedent/consequent objects and add MFs
def create_var(name, universe_min, universe_max, step=1):
    uni = np.arange(universe_min, universe_max+step, step)
    var = ctrl.Antecedent(uni, name)
    return var

# Create variables
vars_ = {}
for vname, (lo, hi) in spec['universes'].items():
    # for categorical small ranges, choose step=1 else step=1 is okay
    vars_[vname] = create_var(vname, lo, hi, step=1)

# Add MFs from spec to each var
def add_mf(var_obj, mf_name, mf_spec):
    if mf_spec['type'] == 'trimf':
        params = mf_spec['params']
        var_obj[mf_name] = fuzz.trimf(var_obj.universe, params)
    elif mf_spec['type'] == 'trap':
        params = mf_spec['params']
        var_obj[mf_name] = fuzz.trapmf(var_obj.universe, params)
    elif mf_spec['type'] == 'singleton':
        # emulate singleton using a tight trimf around the point
        p = mf_spec['params'][0]
        eps = 0.001 * (var_obj.universe.max() - var_obj.universe.min())  # tiny width
        var_obj[mf_name] = fuzz.trimf(var_obj.universe, [p-eps, p, p+eps])
    else:
        raise ValueError("Unknown MF type: "+mf_spec['type'])

# Apply MFs (but for numeric MFs we will later overwrite from GA chromosome)
for vname, mfs in spec['mfs'].items():
    for mf_name, mf_spec in mfs.items():
        # avoid adding numeric MFs now if you want GA to control them — but add initial baseline ones first
        add_mf(vars_[vname], mf_name, mf_spec)

# Recreate rules from templates
rules_list = []
for antecedent, consequent in spec['rules']:
    # build left side: chain conditions with & operator
    left = None
    for feat, mf_name in antecedent.items():
        cond = vars_[feat][mf_name]
        if left is None:
            left = cond
        else:
            left = left & cond
    # build right side (consequent) - assume stroke_risk
    right_feat, right_mf = list(consequent.items())[0]
    # if consequent variable is stroke_risk, ensure it's created
    if 'stroke_risk' not in vars_:
        # create stroke_risk consequent range 0..2
        out = ctrl.Consequent(np.arange(0,3,1), 'stroke_risk')
        # add standard MFs (low/medium/high)
        out['low'] = fuzz.trimf(out.universe, [0,0,1])
        out['medium'] = fuzz.trimf(out.universe, [0,1,2])
        out['high'] = fuzz.trimf(out.universe, [1,2,2])
        vars_['stroke_risk'] = out
    right = vars_['stroke_risk'][right_mf]
    rules_list.append(ctrl.Rule(left, right))

# Build control system once (baseline)
stroke_ctrl = ctrl.ControlSystem(rules_list)
# We won't create simulation here if GA will rebuild per-chromosome; but can create baseline sim:
stroke_sim = ctrl.ControlSystemSimulation(stroke_ctrl)

print("Rebuilt fuzzy system from spec. Rules:", len(rules_list))


Rebuilt fuzzy system from spec. Rules: 18


In [10]:
# ---------- Use spec (from NB3) to build chromosome & encode rules ----------
import numpy as np
import pickle
import json

# If you haven't loaded spec yet, load it:
# with open("fuzzy_system_spec.pkl", "rb") as f:
#     spec = pickle.load(f)

# Sanity: ensure 'spec' exists
try:
    spec
except NameError:
    raise RuntimeError("spec not found — make sure you loaded fuzzy_system_spec.pkl as 'spec' earlier.")

# 1) Define which features GA should optimize (numeric continuous)
numeric_features = ['age', 'bmi', 'avg_glucose_level']

# 2) helper: map real param -> normalized [0,1] relative to universe
def normalize_param(val, lo, hi):
    if hi == lo:
        return 0.0
    return (val - lo) / (hi - lo)

def denormalize_param(norm, lo, hi):
    return lo + norm * (hi - lo)

# 3) Build chromosome initial vector from spec (normalize numeric-MF params)
chromosome_parts = []
numeric_mf_meta = {}  # store meta for rebuilding: feature -> list of (mf_name, lo, hi, original_params)
for feat in numeric_features:
    uni_lo, uni_hi = spec['universes'][feat][0], spec['universes'][feat][1]
    mfs = spec['mfs'][feat]  # dict of mf_name -> {type, params}
    # enforce a deterministic order (sorted by mf name) or prefer specific order (young/middle/old etc.)
    # We'll try to preserve natural order if keys are known:
    preferred_order = ['young','low','normal','middle','elevated','high','old']  # fallback ordering
    mf_names = [name for name in ['young','middle','old','low','normal','high','elevated'] if name in mfs]
    if not mf_names:
        mf_names = sorted(mfs.keys())
    numeric_mf_meta[feat] = []
    for mf_name in mf_names:
        mf_spec = mfs[mf_name]
        params = mf_spec['params']
        # params can be length 3 (trimf) or 4 (trapmf) — normalize each
        norm_params = [normalize_param(p, uni_lo, uni_hi) for p in params]
        chromosome_parts.extend(norm_params)
        numeric_mf_meta[feat].append({
            'mf_name': mf_name,
            'type': mf_spec['type'],
            'params_original': params,
            'uni_lo': uni_lo, 'uni_hi': uni_hi,
            'n_params': len(params)
        })

# Final chromosome (as numpy array in [0,1])
initial_chromosome = np.array(chromosome_parts, dtype=float)
print("Initial chromosome length (numeric-only):", initial_chromosome.size)

# Save meta so we can decode segments reliably
# Build index map: feature -> (start_idx, n_params_total)
index_map = {}
idx = 0
for feat, metas in numeric_mf_meta.items():
    n_params = sum(m['n_params'] for m in metas)
    index_map[feat] = (idx, n_params)
    idx += n_params

print("Index map per feature:", index_map)

# 4) Encode rules templates (from spec['rules']) into list of antecedents & consequent
# We'll produce a list of dicts: {'antecedent': {'age':'old',...}, 'consequent': {'stroke_risk':'high'}}
rule_templates = []
for antecedent, consequent in spec['rules']:
    # keep as-is (they already had this structure in exporter)
    rule_templates.append({'antecedent': antecedent, 'consequent': consequent})

print("Number of rule templates loaded:", len(rule_templates))
# print first few rules to confirm
for r in rule_templates[:5]:
    print(r)

# 5) Provide helper to decode a chromosome back into MF params (real scale)
def decode_numeric_chromosome(chromosome):
    decoded = {}
    for feat, (start, n) in index_map.items():
        seg = chromosome[start:start+n]
        metas = numeric_mf_meta[feat]
        out = []
        j = 0
        for meta in metas:
            np_count = meta['n_params']
            seg_part = seg[j:j+np_count]
            # denormalize back to real units
            real_params = [denormalize_param(val, meta['uni_lo'], meta['uni_hi']) for val in seg_part]
            out.append({'mf_name': meta['mf_name'], 'type': meta['type'], 'params': real_params})
            j += np_count
        decoded[feat] = out
    return decoded

# Quick sanity decode test
decoded_test = decode_numeric_chromosome(initial_chromosome)
for feat, mfs in decoded_test.items():
    print(f"\n{feat}:")
    for m in mfs:
        print(" ", m['mf_name'], m['type'], m['params'])

# Persist meta & initial chromosome for later GA use if you want
import pickle
with open("nb4_chromosome_meta.pkl", "wb") as f:
    pickle.dump({'initial_chromosome': initial_chromosome,
                 'numeric_mf_meta': numeric_mf_meta,
                 'index_map': index_map,
                 'rule_templates': rule_templates,
                 'spec_universes': spec['universes']}, f)

print("Saved nb4_chromosome_meta.pkl with initial chromosome and meta.")


Initial chromosome length (numeric-only): 33
Index map per feature: {'age': (0, 11), 'bmi': (11, 11), 'avg_glucose_level': (22, 11)}
Number of rule templates loaded: 18
{'antecedent': {'age': 'old', 'hypertension': 'yes'}, 'consequent': {'stroke_risk': 'high'}}
{'antecedent': {'age': 'old', 'heart_disease': 'yes'}, 'consequent': {'stroke_risk': 'high'}}
{'antecedent': {'age': 'middle', 'hypertension': 'yes'}, 'consequent': {'stroke_risk': 'medium'}}
{'antecedent': {'age': 'young', 'hypertension': 'yes'}, 'consequent': {'stroke_risk': 'medium'}}
{'antecedent': {'avg_glucose_level': 'high', 'bmi': 'high'}, 'consequent': {'stroke_risk': 'high'}}

age:
  young trap [0.0, 0.0, 25.0, 35.0]
  middle trimf [30.0, 45.0, 60.0]
  old trap [55.00000000000001, 65.0, 100.0, 100.0]

bmi:
  low trap [10.0, 10.0, 18.0, 22.0]
  normal trimf [20.0, 25.0, 30.0]
  high trap [28.0, 32.0, 60.0, 60.0]

avg_glucose_level:
  normal trap [50.0, 50.0, 90.0, 110.0]
  high trap [150.0, 180.0, 300.0, 300.0]
  elevat

In [11]:
# ------------------------------------------------------------
# Build fuzzy system from chromosome (core function for the GA)
# ------------------------------------------------------------
from skfuzzy import control as ctrl
import numpy as np

def build_fuzzy_system_from_chromosome(
    chromosome,
    numeric_mf_meta,
    index_map,
    rule_templates,
    universes
):
    """
    Rebuild a complete fuzzy control system from a GA chromosome.
    This produces a NEW crisp-tuning fuzzy model for evaluation.
    """

    # ---------------------------
    # 1) Decode chromosome → MF params
    # ---------------------------
    decoded = {}
    for feat, (start, count) in index_map.items():
        seg = chromosome[start:start+count]
        metas = numeric_mf_meta[feat]
        out = []
        j = 0
        for meta in metas:
            p_count = meta['n_params']
            sub = seg[j:j+p_count]
            # de-normalize
            real_params = [
                universes[feat][0] + v * (universes[feat][1] - universes[feat][0])
                for v in sub
            ]
            out.append({
                'mf_name': meta['mf_name'],
                'type': meta['type'],
                'params': real_params
            })
            j += p_count
        decoded[feat] = out

    # ---------------------------
    # 2) Create fuzzy Antecedents & Consequents
    # ---------------------------
    fuzzy_vars = {}

    # First: numeric vars
    for feat, mflist in decoded.items():
        lo, hi = universes[feat]
        fuzzy_vars[feat] = ctrl.Antecedent(np.arange(lo, hi+1, 1), feat)

        for mf in mflist:
            name = mf['mf_name']
            params = mf['params']

            if mf['type'] == 'trimf':
                fuzzy_vars[feat][name] = ctrl.trimf(fuzzy_vars[feat].universe, params)

            elif mf['type'] == 'trapmf':
                fuzzy_vars[feat][name] = ctrl.trapmf(fuzzy_vars[feat].universe, params)

    # Second: categorical vars (fixed MFs)
    # NOTE: these DO NOT get optimized by GA.
    categorical_unis = {
        'gender': [0, 1],
        'ever_married': [0, 1],
        'work_type': [0, 4],
        'smoking_status': [0, 2],
        'hypertension': [0, 1],
        'heart_disease': [0, 1],
        'Residence_type': [0, 1],
    }

    # fixed crisp MFs
    for feat, (lo, hi) in categorical_unis.items():
        fuzzy_vars[feat] = ctrl.Antecedent(np.arange(lo, hi+1, 1), feat)
        fuzzy_vars[feat]['yes'] = ctrl.trimf(fuzzy_vars[feat].universe, [0.5, 1, 1])
        fuzzy_vars[feat]['no']  = ctrl.trimf(fuzzy_vars[feat].universe, [0, 0, 0.5])

    # Consequent
    lo, hi = universes['stroke_risk']
    fuzzy_vars['stroke_risk'] = ctrl.Consequent(np.arange(lo, hi+1, 1), 'stroke_risk')
    fuzzy_vars['stroke_risk']['low'] = ctrl.trimf(fuzzy_vars['stroke_risk'].universe, [0, 0, 1])
    fuzzy_vars['stroke_risk']['medium'] = ctrl.trimf(fuzzy_vars['stroke_risk'].universe, [0.5, 1, 1.5])
    fuzzy_vars['stroke_risk']['high'] = ctrl.trimf(fuzzy_vars['stroke_risk'].universe, [1, 2, 2])

    # ---------------------------
    # 3) Rebuild rules from templates
    # ---------------------------
    rules = []
    for rule in rule_templates:
        ant = rule['antecedent']
        cons = rule['consequent']

        antecedent_terms = []
        for feat, mf in ant.items():
            antecedent_terms.append(fuzzy_vars[feat][mf])

        # connect using AND
        big_and = antecedent_terms[0]
        for i in range(1, len(antecedent_terms)):
            big_and = big_and & antecedent_terms[i]

        consequent_feat = list(cons.keys())[0]
        consequent_mf   = list(cons.values())[0]

        rules.append(ctrl.Rule(big_and, fuzzy_vars[consequent_feat][consequent_mf]))

    # ---------------------------
    # 4) Build control system
    # ---------------------------
    system = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(system)

    return sim, fuzzy_vars


In [13]:
# ---------------- GA: fitness + main loop (Notebook 4) ----------------
import numpy as np
import random
import time
import pickle
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import skfuzzy.membership as mf  # <-- هنا استيراد trimf / trapmf

# ---------- sanity: ensure required meta exist ----------
try:
    numeric_mf_meta
    index_map
    rule_templates
    spec
except NameError as e:
    raise RuntimeError("Required meta not found (numeric_mf_meta/index_map/rule_templates/spec). Make sure you loaded nb4_chromosome_meta.pkl and spec.") from e

# build universes dict for build_fuzzy_system_from_chromosome
universes = spec['universes'].copy()
# ensure stroke_risk universe exists
if 'stroke_risk' not in universes:
    universes['stroke_risk'] = [0, 2]

# ---------- create validation set for GA (if not already) ----------
try:
    val_df
    y_val
except NameError:
    X_train_all, X_val_all, y_train_all, y_val_all = train_test_split(
        X_fuzzy, y_fuzzy, test_size=0.30, random_state=42, stratify=y_fuzzy
    )

    val_df = pd.DataFrame(X_val_all, columns=['age','bmi','gender','ever_married','work_type',
                                             'smoking_status','avg_glucose_level','hypertension',
                                             'heart_disease','Residence_type'])
    y_val = np.array(y_val_all)

print("Validation rows:", len(val_df))

# ---------- helper: evaluate a chromosome using sampled rows ----------
def evaluate_fitness_real(chromosome, n_samples=300, verbose=False):
    """Build fuzzy sim from chromosome and evaluate accuracy on random sample of validation set."""
    sim, fuzzy_vars = build_fuzzy_system_from_chromosome(
        chromosome, numeric_mf_meta, index_map, rule_templates, universes
    )

    # choose sample
    if n_samples is None or n_samples >= len(val_df):
        rows = val_df
        ys = y_val
    else:
        sample_idx = np.random.choice(len(val_df), size=n_samples, replace=False)
        rows = val_df.iloc[sample_idx]
        ys = y_val[sample_idx]

    preds = []
    for _, row in rows.iterrows():
        # set inputs expected names: age, bmi, avg_glucose_level, hypertension, heart_disease, smoking_status
        sim.input['age'] = float(row['age'])
        sim.input['bmi'] = float(row['bmi'])
        sim.input['avg_glucose_level'] = float(row['avg_glucose_level'])
        sim.input['hypertension'] = int(row['hypertension'])
        sim.input['heart_disease'] = int(row['heart_disease'])
        sim.input['smoking_status'] = int(row['smoking_status'])
        try:
            sim.compute()
            outv = sim.output['stroke_risk']
        except Exception as e:
            # if simulation fails, fallback to medium
            outv = 1.0
        if outv < 0.5:
            preds.append(0)
        elif outv < 1.5:
            preds.append(1)
        else:
            preds.append(2)
    acc = accuracy_score(ys, preds)
    if verbose:
        print("Sample eval acc:", acc)
    return acc

# ---------- GA operators ----------
def select_elites(population, fitnesses, elite_size):
    idx = np.argsort(fitnesses)[::-1]
    return population[idx[:elite_size]]

def crossover(parent1, parent2):
    # two-point crossover for more mixing
    L = len(parent1)
    a = np.random.randint(1, L//2)
    b = np.random.randint(L//2, L-1)
    c1 = np.concatenate([parent1[:a], parent2[a:b], parent1[b:]])
    c2 = np.concatenate([parent2[:a], parent1[a:b], parent2[b:]])
    return c1, c2

def mutate(chromosome, mutation_rate, sigma=0.06):
    for i in range(len(chromosome)):
        if np.random.rand() < mutation_rate:
            chromosome[i] = np.clip(chromosome[i] + np.random.normal(0, sigma), 0, 1)
    return chromosome

# ---------- GA hyperparameters ----------
population_size = 28
num_generations = 25
elite_size = 4
mutation_rate = 0.12
sample_size_per_eval = 350  # use sampled rows to speed up per-eval

# ---------- initialize population ----------
try:
    population
except NameError:
    if 'initial_chromosome' in globals():
        center = initial_chromosome
    else:
        center = np.random.rand(sum(v[1] for v in index_map.values()))
    population = []
    for i in range(population_size):
        pop_ind = np.clip(center + np.random.normal(0, 0.12, size=center.shape), 0, 1)
        population.append(pop_ind)
    population = np.array(population)

print("Population shape:", population.shape)

# ---------- GA main loop ----------
fitness_history = []
best_individual = None
best_fitness = -1
start = time.time()

for gen in range(num_generations):
    fitnesses = np.zeros(len(population))
    for i, ind in enumerate(population):
        fitnesses[i] = evaluate_fitness_real(ind, n_samples=sample_size_per_eval)
    gen_best_idx = np.argmax(fitnesses)
    gen_best = fitnesses[gen_best_idx]
    fitness_history.append(gen_best)
    if gen_best > best_fitness:
        best_fitness = gen_best
        best_individual = population[gen_best_idx].copy()
    print(f"Gen {gen+1}/{num_generations} | best: {gen_best:.4f} | mean: {fitnesses.mean():.4f}")

    elites = select_elites(population, fitnesses, elite_size)

    new_pop = list(elites.copy())
    while len(new_pop) < population_size:
        p1, p2 = random.sample(list(elites), 2)
        c1, c2 = crossover(p1, p2)
        new_pop.append(mutate(c1.copy(), mutation_rate))
        if len(new_pop) < population_size:
            new_pop.append(mutate(c2.copy(), mutation_rate))
    population = np.array(new_pop)

end = time.time()
print("GA finished in", round(end-start,1), "seconds")
print("Best fitness (sampled):", best_fitness)

# ---------- final evaluation ----------
final_acc = evaluate_fitness_real(best_individual, n_samples=None, verbose=True)
print("Final accuracy on full validation set:", final_acc)

# ---------- save best chromosome + optimized spec ----------
best_chrom = best_individual.copy()
spec_opt = spec.copy()
spec_opt['mfs'] = pickle.loads(pickle.dumps(spec_opt['mfs']))  # deep copy

decoded_best = {}
for feat, (start, count) in index_map.items():
    seg = best_chrom[start:start+count]
    metas = numeric_mf_meta[feat]
    j = 0
    for meta in metas:
        np_count = meta['n_params']
        seg_part = seg[j:j+np_count]
        real_params = [universes[feat][0] + v*(universes[feat][1]-universes[feat][0]) for v in seg_part]
        spec_opt['mfs'][feat][meta['mf_name']]['params'] = [float(x) for x in real_params]
        j += np_count

with open("fuzzy_system_spec_optimized.pkl", "wb") as f:
    pickle.dump({'spec_opt': spec_opt, 'best_chromosome': best_chrom, 'fitness_history': fitness_history}, f)

print("Saved fuzzy_system_spec_optimized.pkl (optimized MF params + best chromosome).")

# ---------- plot fitness history ----------
plt.figure(figsize=(7,4))
plt.plot(fitness_history, marker='o')
plt.title("GA progress (best fitness per generation)")
plt.xlabel("Generation")
plt.ylabel("Best fitness (sampled val)")
plt.grid(True)
plt.show()


Validation rows: 1533
Population shape: (28, 33)


AttributeError: module 'skfuzzy.control' has no attribute 'trimf'